# Practical 1: Real valued Max Ones solution

In [ ]:
!pip install deap

In [ ]:
import random
import numpy
import matplotlib.pyplot as plt
from deap import algorithms, base, creator, tools

In [ ]:
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

We need to change the representation of the individual from being boolean to being real, and change the way that we initialize the individual, so that it's a random number (here between 0 and 1) rather than a 0 or 1.

In [ ]:
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("attr_real", random.random)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_real, n=10)

A common mistake is to define attr_real like this.
> toolbox.register("attr_real", random.random() )  

But you should not have the brackets after real. When you register a function you only include the function name, followed by any arguments (see. e.g. the original random.randit). Here random.random has no arguments.

In [ ]:
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("select", tools.selTournament, tournsize=3)

def evalOneMax(individual):
    return (sum(individual),)

toolbox.register("evaluate", evalOneMax)

Next we need to change crossover. We could still use uniform flips in this case, but it would be a poor choice because it is quite disruptive and thus does not make as much sense when we are not dealing with binary flips. Instead, let's change it to the commonly used one-point crossover. Because we needed to remove the probability of crossover per gene here, we now add a probability of crossover to the main body of the code below.

In [ ]:
toolbox.register("mate", tools.cxOnePoint)

We also need to change the mutation operator. There are many options that we could use when it comes to continuous values. Here, I have chosen Guassian mutation. Notice that mutation can increase or decrease the value of a gene, and that we have not implemented boundary control.

In [ ]:
toolbox.register("mutate", tools.mutGaussian, mu=0.0, sigma=0.2, indpb=0.1)

In [ ]:
logbook = tools.Logbook()
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", numpy.mean)
stats.register("std", numpy.std)
stats.register("min", numpy.min)
stats.register("max", numpy.max)

In [ ]:
pop = toolbox.population(n=100)
fitnesses = list(map(toolbox.evaluate, pop))
for ind, fit in zip(pop, fitnesses):
    ind.fitness.values = fit

In [ ]:
NGEN = 100
CXProb = 0.5 # probability of crossover per proposed pair of parents

for g in range(NGEN):
    print("-- Generation %i --" % g)

    offspring = toolbox.select(pop, len(pop))
    offspring = list(map(toolbox.clone, offspring))

    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < CXProb:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    for mutant in offspring:
        toolbox.mutate(mutant)
        del mutant.fitness.values

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
         ind.fitness.values = fit

    pop[:] = offspring

    record = stats.compile(pop)
    logbook.record(gen=g, **record)

In [ ]:
logbook.header = "gen", "avg", "std", "min", "max"
gen = logbook.select("gen")
avgs = logbook.select("avg")
stds = logbook.select("std")

In [ ]:
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
plt.rc('legend', fontsize=14)
fig, ax1 = plt.subplots()
#line1 = ax1.plot(gen, avgs)
line1 = ax1.errorbar(gen, avgs, yerr=stds, errorevery=2)
ax1.set_xlabel("Generation")
ax1.set_ylabel("Mean Fitness")

In [ ]:
print(logbook)

This is an unusual looking fitness plot, because it is gradual the whole way. Also notice that we do not have an upper bound for this problem, so the fitness will continue like this.

In [ ]:
print(pop[0])

# Adding your own mutation operator

You were next asked to add your own mutation operator, and should've produced something like we see below.

In [ ]:
def mutUniformReal(individual, indpb=0.2 ):
    size = len(individual)
    for i in range(size):
        if random.random() < indpb:
            individual[i] = individual[i] + random.uniform(-1.0,1.0)
    return individual,

toolbox.register("mutate", mutUniformReal, indpb=0.2)

In [ ]:
toolbox.register("mutate", mutUniformReal,indpb=0.1)